In [1]:
from ash import *

ashpath: /Users/tom/Documents/ucl/projects/ash-fork/ash
Sys path: ['/Users/tom/Documents/ucl/projects/ash-fork/ash', '/Users/tom/.local/share/uv/python/cpython-3.11.12-macos-aarch64-none/lib/python311.zip', '/Users/tom/.local/share/uv/python/cpython-3.11.12-macos-aarch64-none/lib/python3.11', '/Users/tom/.local/share/uv/python/cpython-3.11.12-macos-aarch64-none/lib/python3.11/lib-dynload', '', '/Users/tom/Documents/ucl/projects/ash-fork/.venv/lib/python3.11/site-packages', '__editable__.ash-0.95.finder.__path_hook__']


/Users/tom/Documents/ucl/projects/ash-fork/.venv/lib/python3.11/site-packages/pennylane/__init__.py:21: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


--------------------------------------------------------------------------------
--------------------------------------------------------------------------------
                                           ASH                                            
                              A MULTISCALE MODELLING PROGRAM                              
                                        Version: 0.9dev                                        
                               Git commit version: Unknown                                
--------------------------------------------------------------------------------
--------------------------------------------------------------------------------
ASH path: /Users/tom/Documents/ucl/projects/ash-fork/ash
Python version: 3.11.12
Python interpreter: /Users/tom/Documents/ucl/projects/ash-fork/.venv/bin/python

ASH Settings after reading defaults and ~/ash_user_settings.ini : 
See https://ash.readthedocs.io/en/latest/basics.html#ash-settings on how to ch

In [2]:
BASIS = 'sto-3g'
XC = 'b3lyp'
CHARGE = 0
MULT = 1

In [3]:
#Defining fragment
frag = Fragment(xyzfile="system_aftersolvent.xyz", charge=CHARGE, mult=MULT)


--------------------------------------------------------------------------------
                                New ASH fragment                                
--------------------------------------------------------------------------------

ASH Fragment creation
Reading coordinates from XYZ file 'system_aftersolvent.xyz' into fragment.
Creating/Updating fragment attributes...
Number of Atoms in fragment: 2637
Formula: O878H1758C1
Label: system_aftersolvent
Charge: 0 Mult: 1

--------------------------------------------------------------------------------


In [4]:
xyz_list = [i for i in zip(frag.elems, frag.coords)]

lines = [str(len(xyz_list)), '']
for symbol, coords in xyz_list:
    line = f"{symbol} {coords[0]} {coords[1]} {coords[2]}"
    lines.append(line)

xyz_string = "\n".join(lines)

In [5]:
qm_atoms = [0,1,2,3,4,5]

In [6]:
qm_xyz_list = [
    (frag.elems[i], frag.coords[i])
    for i in qm_atoms
]

lines = [str(len(qm_xyz_list)), '']
for symbol, coords in qm_xyz_list:
    line = f"{symbol} {coords[0]} {coords[1]} {coords[2]}"
    lines.append(line)

qm_xyz_string = "\n".join(lines)
print(qm_xyz_string)

6

O 0.0 -1.433 0.0
H -0.962 -1.67 0.0
C 0.0 0.0 0.0
H -0.49 0.417 0.886
H 1.04 0.331 0.0
H -0.49 0.417 -0.886


In [7]:
N_ACT = 2

In [8]:
nbed_theory = NbedTheory(
    geometry=qm_xyz_string,
    n_active_atoms=N_ACT,
    basis=BASIS,
    xc_functional=XC,
    projector='mu',
    localization='spade'
)



                     #####################################                      
                     #                                   #                      
                     #     NbedTheory initialization     #                      
                     #                                   #                      
                     #####################################                      


In [ ]:
# water_xml = "/opt/homebrew/Caskroom/miniconda/base/envs/ash-conda/lib/python3.11/site-packages/openmm/app/data/amber14/tip3p.xml"
water_xml = "amber14/tip3p.xml"

frozen_atoms=listdiff(frag.allatoms,qm_atoms)

openmm_theory = OpenMMTheory(
    xmlfiles=["openff_LIG.xml", water_xml], 
    pdbfile="system_aftersolvent.pdb", 
    periodic=True, 
    autoconstraints=None,
    rigidwater=False
)



                           #########################                            
                           #                       #                            
                           #     OpenMM Theory     #                            
                           #                       #                            
                           #########################                            
OpenMM CPU threads set to: 1
Imported OpenMM library version: 8.3.1

--------------------------------------------------------------------------------
                             Defining OpenMM object                             
--------------------------------------------------------------------------------

Printlevel: 2
No automatic constraints
AutoConstraint setting: None
Rigidwater constraints: False
Hydrogenmass option: 1.5 Da
Using platform: CPU

--------------------------------------------------------------------------------
                            Setting up force fields.

In [17]:
qmmm_theory = QMMMTheory(
    qm_theory = nbed_theory,
    mm_theory = openmm_theory,
    fragment = frag,
    qm_charge = CHARGE,
    qm_mult = MULT,
    qmatoms = qm_atoms,
    printlevel = 3,
)



                            ########################                            
                            #                      #                            
                            #     QM/MM Theory     #                            
                            #                      #                            
                            ########################                            
QM-theory: NbedTheory
MM-theory: OpenMMTheory
All atoms in fragment: 2637
QM region (6 atoms): [0, 1, 2, 3, 4, 5]
MM region (2631 atoms)
QM/MM object selected to use 1 cores
Embedding: elstat
No atomcharges list passed to QMMMTheory object
Getting system charges from OpenMM object
QM-region coordinates (before linkatoms):
   0    O   0.08085499   -1.50934519    0.00040741
   1    H  -1.03625493   -1.69626354    0.00014052
   2    C  -0.00025929    0.00470257    0.00065451
   3    H  -0.51201348    0.45699428    0.92613640
   4    H   1.08013612    0.35033553    0.00051612
   5    H  -0.

In [ ]:


MolecularDynamics(
    fragment=frag,
    theory=qmmm_theory,
    timestep=0.001,
    simulation_steps=25,
    traj_frequency=1,
    temperature=300,
    # integrator='LangevinIntegrator',
    coupling_frequency=1,
    charge=CHARGE,
    mult=MULT
)

Input geometry is not an existing file. Assumng raw xyz input.Input geometry: 6

O 0.0 -1.433 0.0
H -0.962 -1.67 0.0
C 0.0 0.0 0.0
H -0.49 0.417 0.886
H 1.04 0.331 0.0
H -0.49 0.417 -0.886Input geometry is not an existing file. Assumng raw xyz input.Input geometry: 6

O 0.0 -1.433 0.0
H -0.962 -1.67 0.0
C 0.0 0.0 0.0
H -0.49 0.417 0.886
H 1.04 0.331 0.0
H -0.49 0.417 -0.886



                     ######################################                     
                     #                                    #                     
                     #     OpenMM MD wrapper function     #                     
                     #                                    #                     
                     ######################################                     


              ####################################################              
              #                                                  #              
              #     OpenMM Molecular Dynamics Initialization     #              
              #                                                  #              
              ####################################################              
Analyzing theory input to OpenMM_MDclass
This is an QMMMTheory object
Turning on externalforce option.
Added force

--------------------
MD system parameters
--------------------
Tempera

Global RKS: -114.1760471885187Indices of embedded electrons:[0 1 2 3 4][5 6 7 8]DFT potential average 0.4563204852074258.Input geometry is not an existing file. Assumng raw xyz input.Input geometry: 6

O 0.0 -1.433 0.0
H -0.962 -1.67 0.0
C 0.0 0.0 0.0
H -0.49 0.417 0.886
H 1.04 0.331 0.0
H -0.49 0.417 -0.886Embedded scf energy MU_SHIFT: -43.64476878524836, converged: TrueOrbital indices for embedded system: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]Orbital indices removed from embedded system: [10, 11, 12, 13]V emb mean mu: 149884.79109461783RHF energy: -113.81471910858309Warning: <pyscf.gto.mole.Mole object at 0x305e81e90> must be initialized before calling SCF.
Initialize <pyscf.gto.mole.Mole object at 0x305e81e90> in <pyscf.dft.rks.RKS object at 0x307d61d10>
Embedding complete.Embedded CCSD energy: -0.05792193661898626

MM elements: ['O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', '

Input geometry is not an existing file. Assumng raw xyz input.Input geometry: 6

O 0.0 -1.433 0.0
H -0.962 -1.67 0.0
C 0.0 0.0 0.0
H -0.49 0.417 0.886
H 1.04 0.331 0.0
H -0.49 0.417 -0.886Input geometry is not an existing file. Assumng raw xyz input.Input geometry: 6

O 0.0 -1.433 0.0
H -0.962 -1.67 0.0
C 0.0 0.0 0.0
H -0.49 0.417 0.886
H 1.04 0.331 0.0
H -0.49 0.417 -0.886

...

MM elements: ['O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', '

Input geometry is not an existing file. Assumng raw xyz input.Input geometry: 6

O 0.0 -1.433 0.0
H -0.962 -1.67 0.0
C 0.0 0.0 0.0
H -0.49 0.417 0.886
H 1.04 0.331 0.0
H -0.49 0.417 -0.886Input geometry is not an existing file. Assumng raw xyz input.Input geometry: 6

O 0.0 -1.433 0.0
H -0.962 -1.67 0.0
C 0.0 0.0 0.0
H -0.49 0.417 0.886
H 1.04 0.331 0.0
H -0.49 0.417 -0.886


---------------------------------------------------------------------------
Time to calculate step (openmmobject sim step): 0.029 seconds, 0.0 minutes.
---------------------------------------------------------------------------

--------------------------------------------------------------------
Time to calculate step (Total sim step): 0.636 seconds, 0.0 minutes.
--------------------------------------------------------------------
Step: 24

----------------------------------------------------------------------
Time to calculate step (get OpenMM state): 0.008 seconds, 0.0 minutes.
----------------------------------------------------------------------

------------------------------------------------------------------------
Time to calculate step (get current_coords): 0.000 seconds, 0.0 minutes.
------------------------------------------------------------------------
  ------------RUNNING QM/MM MODULE------------- 
QM Module: NbedTheory
MM Module: OpenMMTheory
Charge provided from QMMM

Global RKS: -114.1760471885187Indices of embedded electrons:[0 1 2 3 4][5 6 7 8]DFT potential average 0.4563204852074258.Input geometry is not an existing file. Assumng raw xyz input.Input geometry: 6

O 0.0 -1.433 0.0
H -0.962 -1.67 0.0
C 0.0 0.0 0.0
H -0.49 0.417 0.886
H 1.04 0.331 0.0
H -0.49 0.417 -0.886Embedded scf energy MU_SHIFT: -43.64476878524836, converged: TrueOrbital indices for embedded system: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]Orbital indices removed from embedded system: [10, 11, 12, 13]V emb mean mu: 149884.79109461783RHF energy: -113.81471910858309Warning: <pyscf.gto.mole.Mole object at 0x301ae24d0> must be initialized before calling SCF.
Initialize <pyscf.gto.mole.Mole object at 0x301ae24d0> in <pyscf.dft.rks.RKS object at 0x307d3fa10>
Embedding complete.Embedded CCSD energy: -0.05792193661898626

MM elements: ['O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', '

In [12]:
Optimizer(
    fragment=frag, 
    theory=qmmm_theory, 
    ActiveRegion=True, 
    actatoms=qm_atoms
)

geometric-optimize called with the following command line:
/Users/tom/Documents/ucl/projects/ash-fork/.venv/lib/python3.11/site-packages/ipykernel_launcher.py --f=/Users/tom/Library/Jupyter/runtime/kernel-v368d81c2b4ed7bef5302405d67a8781c97f5607f8.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    
              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%%%%%%/  



                 #############################################                  
                 #                                           #                  
                 #     geomeTRICOptimizer initialization     #                  
                 #                                           #                  
                 #############################################                  
Creating optimizer object
List of active atoms provided. Setting ActiveRegion to True

force_coordsystem is False.
No convergence settings by user. Using default criteria (same as ORCA)
Printlevel: 2
Coordinate system:  hdlc
Max iterations:  250
Frozen atoms: []
Active Region: True
Number of active atoms: 6
TS Optimization: False
Hessian Option: None
Convergence criteria: {'convergence_energy': 5e-06, 'convergence_grms': 0.0001, 'convergence_gmax': 0.0003, 'convergence_drms': 0.002, 'convergence_dmax': 0.004, 'convergence_cmax': 0.01}


--------------------------------------------------

/Users/tom/Documents/ucl/projects/ash-fork/.venv/lib/python3.11/site-packages/pyscf/dft/libxc.py:512: UserWarning: Since PySCF-2.3, B3LYP (and B3P86) are changed to the VWN-RPA variant, corresponding to the original definition by Stephens et al. (issue 1480) and the same as the B3LYP functional in Gaussian. To restore the VWN5 definition, you can put the setting "B3LYP_WITH_VWN5 = True" in pyscf_conf.py
  warnings.warn('Since PySCF-2.3, B3LYP (and B3P86) are changed to the VWN-RPA variant, '
Global RKS: -114.1760471885187/Users/tom/Documents/ucl/projects/ash-fork/.venv/lib/python3.11/site-packages/nbed/localizers/spade.py:84: RuntimeWarning: divide by zero encountered in matmul
  linalg.fractional_matrix_power(ao_overlap, 0.5) @ occupied_orbitals
/Users/tom/Documents/ucl/projects/ash-fork/.venv/lib/python3.11/site-packages/nbed/localizers/spade.py:84: RuntimeWarning: overflow encountered in matmul
  linalg.fractional_matrix_power(ao_overlap, 0.5) @ occupied_orbitals
/Users/tom/Document

MM elements: ['O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', '

Step    0 : Gradient = 2.658e-02/4.712e-02 (rms/max) Energy = -127.1225630857
Hessian Eigenvalues: 3.18404e-02 5.00000e-02 5.00000e-02 ... 2.35819e-01 2.67782e-01 3.41385e-01
Input geometry is not an existing file. Assumng raw xyz input.Input geometry: 6

O 0.0 -1.433 0.0
H -0.962 -1.67 0.0
C 0.0 0.0 0.0
H -0.49 0.417 0.886
H 1.04 0.331 0.0
H -0.49 0.417 -0.886Input geometry is not an existing file. Assumng raw xyz input.Input geometry: 6

O 0.0 -1.433 0.0
H -0.962 -1.67 0.0
C 0.0 0.0 0.0
H -0.49 0.417 0.886
H 1.04 0.331 0.0
H -0.49 0.417 -0.886

...

Global RKS: -114.1760471885187Indices of embedded electrons:[0 1 2 3 4][5 6 7 8]DFT potential average 0.4563204852074258.Input geometry is not an existing file. Assumng raw xyz input.Input geometry: 6

O 0.0 -1.433 0.0
H -0.962 -1.67 0.0
C 0.0 0.0 0.0
H -0.49 0.417 0.886
H 1.04 0.331 0.0
H -0.49 0.417 -0.886Embedded scf energy MU_SHIFT: -43.64476878524836, converged: TrueOrbital indices for embedded system: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]Orbital indices removed from embedded system: [10, 11, 12, 13]V emb mean mu: 149884.79109461783RHF energy: -113.81471910858309Warning: <pyscf.gto.mole.Mole object at 0x163a057d0> must be initialized before calling SCF.
Initialize <pyscf.gto.mole.Mole object at 0x163a057d0> in <pyscf.dft.rks.RKS object at 0x165da4410>
Embedding complete.Embedded CCSD energy: -0.05792193661898626

MM elements: ['O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', '

Step  249 : Displace = 9.663e-05/1.522e-04 (rms/max) Trust = 1.000e-04 (=) Grad = 2.658e-02/4.714e-02 (rms/max) E (change) = -127.1226170638 (-9.665e-08) Quality = 0.003
Eigenvalues below 1.0000e-05 (-5.0796e-03) - returning guess
Hessian Eigenvalues: 3.18396e-02 5.00000e-02 5.00000e-02 ... 2.03452e-01 2.12776e-01 2.24006e-01


Time for setup 4: 0.15256404876708984

-------------------------------------------------------------
Time to calculate step (QM step): 0.592 seconds, 0.0 minutes.
-------------------------------------------------------------

----------------------------------------------------------------------
Time to calculate step (QMpcgrad prepare): 0.000 seconds, 0.0 minutes.
----------------------------------------------------------------------

--------------------------------------------------------------------------
Time to calculate step (linkatomgrad prepare): 0.000 seconds, 0.0 minutes.
--------------------------------------------------------------------------

----------------------------------------------------------------------------
Time to calculate step (QM/MM gradient prepare): 0.000 seconds, 0.0 minutes.
----------------------------------------------------------------------------
Using OpenMM theory as part of QM/MM.
Using MM on full system. Charges for QM region [0, 1, 2, 3, 4, 5]

NameError: name 'exit' is not defined